# AlexNet 垃圾分类：Colab 一键运行全部实验

1. 菜单 **代码执行程序 → 更改运行时类型 → T4 GPU**
2. **代码执行程序 → 全部运行**

内容：无泄漏的数据划分 → 两个 HOG 基线 → 4 种 AlexNet 配置 × 3 个随机种子 → Grad-CAM → 汇总 → 下载 `results.zip`。T4 上大约 1 小时。

**断线不丢结果**：`SAVE_TO_DRIVE = True` 时结果直接写进 Google Drive 的 `alexnet-waste-results/`（会弹出授权窗口）。断线后重新 *全部运行*，已完成的实验会自动跳过。

In [ ]:
SAVE_TO_DRIVE = True   # 结果保存到 Google Drive，断线后可以接着跑
SEEDS = [0, 1, 2]
PRESETS = ["course", "scratch", "scratch_bn", "finetune"]

!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. 克隆仓库并安装

In [ ]:
import os
%cd /content
if os.path.exists("alexnet-waste-classification"):
    !git -C alexnet-waste-classification pull -q --ff-only
else:
    !git clone -q https://github.com/weirdquantum/alexnet-waste-classification.git
%cd /content/alexnet-waste-classification
!pip install -q -e ".[dev]"
!python -m pytest -q

In [ ]:
# 本次实验的输出目录（仓库自带的 results/ 是旧结果，不使用也不修改）
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/alexnet-waste-results"
else:
    OUT = "/content/results"
os.makedirs(OUT, exist_ok=True)
print("results ->", OUT)

## 2. 数据：下载 TrashNet，去重后按物体分组划分

- 删除内容完全相同、却被标成不同类别的图片
- 用 ImageNet AlexNet 特征把同一物体的多张照片（同类、余弦相似度 ≥ 0.85）归为一组
- 按组做分层划分（70 / 15 / 15），同一物体不会同时出现在训练集和测试集

In [ ]:
!mkdir -p data/download
![ -f data/download/dataset-resized.zip ] || wget -q -O data/download/dataset-resized.zip https://github.com/garythung/trashnet/raw/master/data/dataset-resized.zip
!python scripts/prepare_trashnet.py

# 第一次运行时把划分存进输出目录；之后（例如断线重连）复用同一个划分，保证所有实验用的是同一批数据
SPLIT_FILES = ["train.json", "val.json", "test.json", "split_summary.json"]
os.makedirs(f"{OUT}/split", exist_ok=True)
for name in SPLIT_FILES:
    if os.path.exists(f"{OUT}/split/{name}"):
        !cp "{OUT}/split/{name}" data/trashnet/{name}
    else:
        !cp data/trashnet/{name} "{OUT}/split/{name}"
!cat data/trashnet/split_summary.json

## 3. 传统基线：HOG + PCA + 决策树 / RBF-SVM（CPU，约 3 分钟）

In [ ]:
if not (os.path.exists(f"{OUT}/hog_tree/metrics.json") and os.path.exists(f"{OUT}/hog_svm/metrics.json")):
    !python scripts/train_baseline.py --out {OUT}
else:
    print("baselines done, skipping")

## 4. AlexNet：4 种配置 × 3 个随机种子

| preset | 说明 |
|---|---|
| `course` | 原课程配方：192 px、像素 /255、无增强、Adam 1e-3、PyTorch 默认初始化 |
| `scratch` | 从零训练 + 改进配方（ImageNet 归一化、数据增强、AdamW、label smoothing、warmup + cosine） |
| `scratch_bn` | 同上，卷积后加 BatchNorm |
| `finetune` | ImageNet 预训练 AlexNet 微调 |

按种子从外到内循环：先跑完 seed 0 的全部配置，再跑 seed 1、2。已有 `metrics.json` 的实验会跳过。

In [ ]:
for seed in SEEDS:
    for preset in PRESETS:
        out = f"{OUT}/{preset}/seed{seed}"
        if os.path.exists(f"{out}/metrics.json"):
            print(f"{preset} seed {seed}: done, skipping")
            continue
        print(f"===== {preset} seed {seed} =====", flush=True)
        os.makedirs(out, exist_ok=True)
        !python -u scripts/train.py --preset {preset} --seed {seed} --num-workers 2 --out "{out}" 2>&1 | tee "{out}/train.log" | grep -E "device=|best epoch|early stopping|Error|Traceback|epoch +[0-9]*0 " 

## 5. Grad-CAM：每类最有把握的正确预测和错误预测

In [ ]:
for preset in ["finetune", "scratch"]:
    !python scripts/gradcam_grid.py --checkpoint {OUT}/{preset}/seed0/best.pt

## 6. 汇总

In [ ]:
!python scripts/summarize.py --results-dir "{OUT}"
from IPython.display import Image, display
for name in ["finetune/seed0", "scratch/seed0", "scratch_bn/seed0", "course/seed0", "hog_svm"]:
    for fig, width in [("confusion_matrix.png", 420), ("curves.png", 720), ("gradcam.png", 420)]:
        path = f"{OUT}/{name}/{fig}"
        if os.path.exists(path):
            print(path)
            display(Image(path, width=width))

## 7. 下载结果（不含模型权重）

把 `results.zip` 放到项目文件夹里交给 Claude 即可。

In [ ]:
!cd "{OUT}" && zip -q -r /content/results.zip . -x "*.pt" "*.joblib"
from google.colab import files
files.download("/content/results.zip")

## 8.（可选）下载微调模型权重

约 230 MB，用于发布到 GitHub Release，别人不用训练就能直接用 `scripts/predict.py`。

In [ ]:
!cp "{OUT}/finetune/seed0/best.pt" /content/alexnet_finetune_trashnet.pt
from google.colab import files
files.download("/content/alexnet_finetune_trashnet.pt")